# Attention is all you need
### By: Devansh Sharma

##  Imports

In [54]:
!pip install -q gradio transformers

import torch
import numpy as np
import gradio as gr
import plotly.express as px
import plotly.graph_objects as go
from transformers import BertTokenizer, BertModel

## Loading Models


In [69]:
# Load pre-trained BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased', output_attentions=True)

def get_huggingface_attention(text, layer=0, head=0, remove_sinks=False):
    if not text.strip():
        return None, None

    # 1. Tokenize the input
    inputs = tokenizer(text, return_tensors="pt")
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    # 2. Pass through the model
    with torch.no_grad():
        outputs = model(**inputs)

    # 3. Extract the exact attention matrix
    attention_matrix = outputs.attentions[layer][0, head, :, :].numpy()

    # 4. TOGGLE ATTENTION SINKS
    if remove_sinks and len(tokens) > 2:
        tokens = tokens[1:-1]
        attention_matrix = attention_matrix[1:-1, 1:-1]

    return tokens, attention_matrix

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Visualization

In [70]:
def build_hf_visuals(text, layer, head, remove_sinks):
    tokens, attention_matrix = get_huggingface_attention(text, layer, head, remove_sinks)
    if tokens is None:
        return None, None

    # --- 1. THE HEATMAP ---
    fig_heat = px.imshow(
        attention_matrix, x=tokens, y=tokens,
        labels=dict(x="Key (Attended to)", y="Query (Attending from)", color="Weight"),
        color_continuous_scale="Viridis",
        text_auto=".2f", aspect="auto"
    )
    fig_heat.update_xaxes(side="top")

    # --- 2. THE UPGRADED NETWORK GRAPH ---
    fig_net = go.Figure()
    y_positions = list(range(len(tokens)))[::-1]

    node_sizes = np.sum(attention_matrix, axis=0) * 20 + 10

    for i in range(len(tokens)):
        for j in range(len(tokens)):
            weight = float(attention_matrix[i, j])
            if weight > 0.03:
                fig_net.add_trace(go.Scatter(
                    x=[0, 0.5, 1],
                    y=[y_positions[i], (y_positions[i] + y_positions[j])/2, y_positions[j]],
                    mode="lines",
                    line=dict(width=weight * 8, color=f"rgba(100, 200, 255, {weight})"),
                    line_shape='spline',
                    hoverinfo="text",
                    text=f"{tokens[i]} → {tokens[j]} (Weight: {weight:.2f})",
                    showlegend=False
                ))

    fig_net.add_trace(go.Scatter(
        x=[0]*len(tokens), y=y_positions, mode="markers+text",
        marker=dict(size=25, color="#ff4b4b", line=dict(width=2, color="white")),
        text=tokens, textposition="middle left",
        hoverinfo="text", hovertext=[f"Query: {t}" for t in tokens], showlegend=False
    ))

    fig_net.add_trace(go.Scatter(
        x=[1]*len(tokens), y=y_positions, mode="markers+text",
        marker=dict(size=node_sizes, color="#0068c9", line=dict(width=2, color="white")),
        text=tokens, textposition="middle right",
        hoverinfo="text", hovertext=[f"Key: {t}" for t in tokens], showlegend=False
    ))

    fig_net.update_layout(
        title=f"Pre-trained Attention Routing (BERT Layer {layer}, Head {head})",
        xaxis=dict(visible=False, range=[-0.3, 1.3]),
        yaxis=dict(visible=False),
        height=600, margin=dict(l=0, r=0, t=40, b=0),
        plot_bgcolor="#0e1117",
        paper_bgcolor="#0e1117",
        font=dict(color="white")
    )

    return fig_heat, fig_net

## UI and Deployment

In [71]:
with gr.Blocks(theme=gr.themes.Monochrome()) as demo:
    gr.Markdown("# Hugging Face Attention Visualizer")
    gr.Markdown("Using a pre-trained `bert-base-uncased` model to show real semantic attention mapping.")

    with gr.Row():
        text_input = gr.Textbox(label="Input Sentence", value="Arti saw Shashi crying in her room.")
        btn = gr.Button("Extract Attention", variant="primary")

    with gr.Row():
        layer_slider = gr.Slider(minimum=0, maximum=11, step=1, value=0, label="Transformer Layer (Depth)")
        head_slider = gr.Slider(minimum=0, maximum=11, step=1, value=0, label="Attention Head (Subspace)")

    with gr.Row():
        # The new toggle switch
        remove_sinks_check = gr.Checkbox(label="Hide [CLS] & [SEP] Attention Sinks", value=False)

    with gr.Row():
        plot_heat = gr.Plot(label="Attention Matrix")
    with gr.Row():
        plot_net = gr.Plot(label="Curved Attention Routing Graph")

    # Wire everything up
    inputs = [text_input, layer_slider, head_slider, remove_sinks_check]
    outputs = [plot_heat, plot_net]

    btn.click(fn=build_hf_visuals, inputs=inputs, outputs=outputs)
    text_input.submit(fn=build_hf_visuals, inputs=inputs, outputs=outputs)

    # Allow the sliders and checkbox to instantly redraw the graph when moved
    layer_slider.change(fn=build_hf_visuals, inputs=inputs, outputs=outputs)
    head_slider.change(fn=build_hf_visuals, inputs=inputs, outputs=outputs)
    remove_sinks_check.change(fn=build_hf_visuals, inputs=inputs, outputs=outputs)

demo.launch(inline=True)

/tmp/ipykernel_1162/4039539043.py:1: UserWarning:

The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.



It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f38f97194548564288.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Done and Dusted!
### A star on the repo would be amazing. thanks!